In [2]:
import sys
sys.path.append('../src')
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt

with h5py.File('../data/processed/injected_v1/injected_dataset.h5', 'r') as f:
    images = f['images'][:]

metadata = pd.read_parquet('../data/processed/injected_v1/injected_dataset_metadata.parquet')
print(metadata.columns.tolist())
print(len(metadata))

['index', 'label', 'seed', 'field_idx', 'theta_E', 'e1_lens', 'e2_lens', 'source_x', 'source_y', 'R_sersic_source', 'n_sersic_source', 'e1_source', 'e2_source', 'R_sersic_lens_light', 'amp_lens_light']
3988


In [3]:
sys.path.append('../src')
import head_labels
import lens_injector
import importlib
importlib.reload(head_labels)
importlib.reload(lens_injector)
from head_labels import einstein_ring_score, brightness_asymmetry_score, arc_geometry_score, render_two_band_signal, colour_gradient_score
from lenstronomy.SimulationAPI.sim_api import SimAPI

def compute_head_labels_for_row(row, numPix=120, pixel_scale=0.168):
    if row['label'] == 0:
        return {'ring_score': np.nan, 'asymmetry_score': np.nan,
                'colour_score': np.nan, 'elongation_score': np.nan, 'n_components': np.nan}

    kwargs_lens = [{'theta_E': row['theta_E'], 'e1': row['e1_lens'], 'e2': row['e2_lens'],
                     'center_x': 0.0, 'center_y': 0.0}]
    kwargs_lens_light = [{'amp': row['amp_lens_light'], 'R_sersic': row['R_sersic_lens_light'],
                           'n_sersic': 4, 'e1': row['e1_lens'], 'e2': row['e2_lens'],
                           'center_x': 0.0, 'center_y': 0.0}]
    kwargs_source = [{'amp': 100, 'R_sersic': row['R_sersic_source'], 'n_sersic': row['n_sersic_source'],
                       'e1': row['e1_source'], 'e2': row['e2_source'],
                       'center_x': row['source_x'], 'center_y': row['source_y']}]

    kwargs_data = {
        'pixel_scale': pixel_scale, 'exposure_time': 100, 'magnitude_zero_point': 25.0,
        'sky_brightness': 30.0, 'read_noise': 0.001, 'ccd_gain': 2.5, 'psf_type': 'NONE'
    }
    sim_api = SimAPI(numpix=numPix, kwargs_single_band=kwargs_data, kwargs_model={
        'lens_model_list': ['SIE'], 'source_light_model_list': ['SERSIC_ELLIPSE'],
        'lens_light_model_list': ['SERSIC_ELLIPSE']
    })
    image_model = sim_api.image_model_class()

    full_signal = image_model.image(kwargs_lens=kwargs_lens, kwargs_source=kwargs_source, kwargs_lens_light=kwargs_lens_light)
    galaxy_only = image_model.image(kwargs_lens=kwargs_lens, kwargs_source=[{**kwargs_source[0], 'amp': 0}], kwargs_lens_light=kwargs_lens_light)
    isolated_arc = full_signal - galaxy_only

    source_offset_r = np.sqrt(row['source_x']**2 + row['source_y']**2)
    ring_score = einstein_ring_score(row['theta_E'], source_offset_r)
    asymmetry_score = brightness_asymmetry_score(isolated_arc)
    elongation_score, n_components = arc_geometry_score(isolated_arc)

    two_band = render_two_band_signal(kwargs_lens, kwargs_lens_light, kwargs_source,
                                        lens_model_list=['SIE'], numPix=numPix, pixel_scale=pixel_scale,
                                        seed=int(row['seed']))
    colour_score = colour_gradient_score(two_band)

    return {'ring_score': ring_score, 'asymmetry_score': asymmetry_score,
            'colour_score': colour_score, 'elongation_score': elongation_score, 'n_components': n_components}

In [4]:
import time

test_rows = metadata.iloc[:5]
start = time.time()
results = [compute_head_labels_for_row(row) for _, row in test_rows.iterrows()]
elapsed = time.time() - start

for r in results:
    print(r)
print(f"\n{elapsed:.2f}s for 5 rows, ~{elapsed/5:.3f}s/row")

{'ring_score': np.float64(0.7318016554868539), 'asymmetry_score': np.float64(1.6400816020714288), 'colour_score': np.float64(1.1442421034670702), 'elongation_score': np.float64(0.43626385092884457), 'n_components': 1}
{'ring_score': np.float64(0.6823870833348351), 'asymmetry_score': np.float64(1.6653180547551425), 'colour_score': np.float64(1.3123914876186027), 'elongation_score': np.float64(0.23021335923360986), 'n_components': 2}
{'ring_score': np.float64(0.4930193823438507), 'asymmetry_score': np.float64(1.8934154057010386), 'colour_score': np.float64(0.7877492869567225), 'elongation_score': np.float64(0.5925262735262589), 'n_components': 2}
{'ring_score': np.float64(0.6468803626322823), 'asymmetry_score': np.float64(1.6955079221844345), 'colour_score': np.float64(1.1420063458982268), 'elongation_score': np.float64(0.6462523489833043), 'n_components': 2}
{'ring_score': np.float64(0.700800445440912), 'asymmetry_score': np.float64(0.9266058514228125), 'colour_score': np.float64(0.9586

In [5]:
from tqdm import tqdm

all_head_labels = []
for _, row in tqdm(metadata.iterrows(), total=len(metadata), desc="Computing head labels"):
    all_head_labels.append(compute_head_labels_for_row(row))

head_labels_df = pd.DataFrame(all_head_labels)
metadata_with_heads = pd.concat([metadata.reset_index(drop=True), head_labels_df], axis=1)

print(metadata_with_heads.shape)
print(metadata_with_heads[metadata_with_heads['label']==1][['ring_score', 'asymmetry_score', 'colour_score', 'elongation_score']].describe())
print("Non-lens NaN check:", metadata_with_heads[metadata_with_heads['label']==0][['ring_score', 'asymmetry_score', 'colour_score', 'elongation_score']].isna().all().all())

Computing head labels:   0%|          | 0/3988 [00:00<?, ?it/s]

Computing head labels: 100%|██████████| 3988/3988 [00:24<00:00, 160.31it/s]


(3988, 20)
        ring_score  asymmetry_score  colour_score  elongation_score
count  1993.000000      1993.000000   1993.000000       1993.000000
mean      0.582630         1.686795      0.980126          0.516046
std       0.160722         0.193029      0.194986          0.191159
min       0.026451         0.926606      0.550296          0.021917
25%       0.495132         1.559146      0.835618          0.377275
50%       0.606055         1.706673      0.977972          0.526948
75%       0.703808         1.838139      1.116348          0.658690
max       0.844809         1.999800      1.483741          0.960562
Non-lens NaN check: True


In [6]:
metadata_with_heads.to_parquet('../data/processed/injected_v1/injected_dataset_metadata_with_heads.parquet', index=False)
print("Saved.")

Saved.


In [7]:
import torch
import torch.nn as nn
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class MultiHeadLensModel(nn.Module):
    def __init__(self, encoder, feature_dim=640):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Linear(feature_dim, 2)
        self.ring_head = nn.Linear(feature_dim, 1)
        self.asymmetry_head = nn.Linear(feature_dim, 1)
        self.colour_head = nn.Linear(feature_dim, 1)
        self.elongation_head = nn.Linear(feature_dim, 1)

    def forward(self, x):
        features = self.encoder(x)
        return {
            'class_logits': self.classifier(features),
            'ring': torch.sigmoid(self.ring_head(features)).squeeze(-1),
            'asymmetry': torch.nn.functional.softplus(self.asymmetry_head(features)).squeeze(-1),
            'colour': torch.nn.functional.softplus(self.colour_head(features)).squeeze(-1),
            'elongation': torch.sigmoid(self.elongation_head(features)).squeeze(-1),
        }

encoder = timm.create_model('hf_hub:mwalmsley/zoobot-encoder-greyscale-convnext_nano', pretrained=True, in_chans=1, num_classes=0)
model = MultiHeadLensModel(encoder).to(device)

test_input = torch.randn(4, 1, 120, 120).to(device)
output = model(test_input)
for k, v in output.items():
    print(k, v.shape)

class_logits torch.Size([4, 2])
ring torch.Size([4])
asymmetry torch.Size([4])
colour torch.Size([4])
elongation torch.Size([4])


In [8]:
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

metadata_with_heads = pd.read_parquet('../data/processed/injected_v1/injected_dataset_metadata_with_heads.parquet')
labels = metadata_with_heads['label'].values

indices = np.arange(len(images))
train_idx, temp_idx = train_test_split(indices, test_size=0.3, stratify=labels, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, stratify=labels[temp_idx], random_state=42)

head_columns = ['ring_score', 'asymmetry_score', 'colour_score', 'elongation_score']

class MultiHeadLensDataset(Dataset):
    def __init__(self, images, metadata_df):
        self.images = images
        self.labels = metadata_df['label'].values
        self.head_values = metadata_df[head_columns].values

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = torch.from_numpy(self.images[idx]).float().unsqueeze(0)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        heads = torch.tensor(self.head_values[idx], dtype=torch.float32)
        return image, label, heads


train_dataset = MultiHeadLensDataset(images[train_idx], metadata_with_heads.iloc[train_idx])
val_dataset = MultiHeadLensDataset(images[val_idx], metadata_with_heads.iloc[val_idx])
test_dataset = MultiHeadLensDataset(images[test_idx], metadata_with_heads.iloc[test_idx])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(len(train_idx), len(val_idx), len(test_idx))
sample_img, sample_label, sample_heads = train_dataset[0]
print(sample_heads)

2791 598 599
tensor([nan, nan, nan, nan])


In [9]:
def multi_head_loss(outputs, labels, heads, head_names=['ring', 'asymmetry', 'colour', 'elongation']):
    classification_loss = nn.functional.cross_entropy(outputs['class_logits'], labels)

    is_lens_mask = (labels == 1)
    n_lens_in_batch = is_lens_mask.sum().item()

    auxiliary_losses = {}
    if n_lens_in_batch > 0:
        for i, name in enumerate(head_names):
            pred = outputs[name][is_lens_mask]
            target = heads[is_lens_mask, i]
            auxiliary_losses[name] = nn.functional.mse_loss(pred, target)
    else:
        for name in head_names:
            auxiliary_losses[name] = torch.tensor(0.0, device=outputs['class_logits'].device)

    total_auxiliary_loss = sum(auxiliary_losses.values())
    total_loss = classification_loss + 0.5 * total_auxiliary_loss

    return total_loss, classification_loss, auxiliary_losses

In [10]:
sample_batch = next(iter(train_loader))
images_batch, labels_batch, heads_batch = sample_batch
images_batch, labels_batch, heads_batch = images_batch.to(device), labels_batch.to(device), heads_batch.to(device)

with torch.no_grad():
    outputs = model(images_batch)
    total_loss, class_loss, aux_losses = multi_head_loss(outputs, labels_batch, heads_batch)

print("Total loss:", total_loss.item())
print("Classification loss:", class_loss.item())
print("Auxiliary losses:", {k: v.item() for k, v in aux_losses.items()})

Total loss: 1.8946176767349243
Classification loss: 0.7382441759109497
Auxiliary losses: {'ring': 0.03632023558020592, 'asymmetry': 2.1651525497436523, 'colour': 0.06169375404715538, 'elongation': 0.04958048090338707}


In [11]:
head_scale_factors = {'ring': 1.0, 'asymmetry': 2.0, 'colour': 1.5, 'elongation': 1.0}

def multi_head_loss(outputs, labels, heads, head_names=['ring', 'asymmetry', 'colour', 'elongation']):
    classification_loss = nn.functional.cross_entropy(outputs['class_logits'], labels)

    is_lens_mask = (labels == 1)
    n_lens_in_batch = is_lens_mask.sum().item()

    auxiliary_losses = {}
    if n_lens_in_batch > 0:
        for i, name in enumerate(head_names):
            scale = head_scale_factors[name]
            pred = outputs[name][is_lens_mask] / scale
            target = heads[is_lens_mask, i] / scale
            auxiliary_losses[name] = nn.functional.mse_loss(pred, target)
    else:
        for name in head_names:
            auxiliary_losses[name] = torch.tensor(0.0, device=outputs['class_logits'].device)

    total_auxiliary_loss = sum(auxiliary_losses.values())
    total_loss = classification_loss + 0.5 * total_auxiliary_loss

    return total_loss, classification_loss, auxiliary_losses

In [12]:
with torch.no_grad():
    total_loss, class_loss, aux_losses = multi_head_loss(outputs, labels_batch, heads_batch)

print("Total loss:", total_loss.item())
print("Auxiliary losses:", {k: v.item() for k, v in aux_losses.items()})

Total loss: 1.065548300743103
Auxiliary losses: {'ring': 0.03632023558020592, 'asymmetry': 0.5412881374359131, 'colour': 0.027419446036219597, 'elongation': 0.04958048090338707}


In [13]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=1e-5)

def train_one_epoch_multihead(model, loader, optimizer, device):
    model.train()
    total_loss_sum = 0.0
    class_correct = 0
    total = 0
    head_loss_sums = {'ring': 0.0, 'asymmetry': 0.0, 'colour': 0.0, 'elongation': 0.0}

    for images_batch, labels_batch, heads_batch in loader:
        images_batch, labels_batch, heads_batch = images_batch.to(device), labels_batch.to(device), heads_batch.to(device)

        optimizer.zero_grad()
        outputs = model(images_batch)
        loss, class_loss, aux_losses = multi_head_loss(outputs, labels_batch, heads_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        batch_size = images_batch.size(0)
        total_loss_sum += loss.item() * batch_size
        for name, l in aux_losses.items():
            head_loss_sums[name] += l.item() * batch_size

        _, predicted = torch.max(outputs['class_logits'], 1)
        class_correct += (predicted == labels_batch).sum().item()
        total += batch_size

    avg_head_losses = {k: v / total for k, v in head_loss_sums.items()}
    return total_loss_sum / total, class_correct / total, avg_head_losses


def evaluate_multihead(model, loader, device):
    model.eval()
    total_loss_sum = 0.0
    class_correct = 0
    total = 0
    head_loss_sums = {'ring': 0.0, 'asymmetry': 0.0, 'colour': 0.0, 'elongation': 0.0}

    with torch.no_grad():
        for images_batch, labels_batch, heads_batch in loader:
            images_batch, labels_batch, heads_batch = images_batch.to(device), labels_batch.to(device), heads_batch.to(device)
            outputs = model(images_batch)
            loss, class_loss, aux_losses = multi_head_loss(outputs, labels_batch, heads_batch)

            batch_size = images_batch.size(0)
            total_loss_sum += loss.item() * batch_size
            for name, l in aux_losses.items():
                head_loss_sums[name] += l.item() * batch_size

            _, predicted = torch.max(outputs['class_logits'], 1)
            class_correct += (predicted == labels_batch).sum().item()
            total += batch_size

    avg_head_losses = {k: v / total for k, v in head_loss_sums.items()}
    return total_loss_sum / total, class_correct / total, avg_head_losses


In [14]:
n_epochs = 15
best_val_loss = float('inf')

history_multihead = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [],
                       'train_heads': [], 'val_heads': []}

for epoch in range(n_epochs):
    train_loss, train_acc, train_heads = train_one_epoch_multihead(model, train_loader, optimizer, device)
    val_loss, val_acc, val_heads = evaluate_multihead(model, val_loader, device)

    history_multihead['train_loss'].append(train_loss)
    history_multihead['train_acc'].append(train_acc)
    history_multihead['val_loss'].append(val_loss)
    history_multihead['val_acc'].append(val_acc)
    history_multihead['train_heads'].append(train_heads)
    history_multihead['val_heads'].append(val_heads)

    marker = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), '../models/multihead_convnext_nano_best.pt')
        marker = " [NEW BEST]"

    print(f"Epoch {epoch+1}/{n_epochs} - train_loss: {train_loss:.4f}, train_acc: {train_acc:.4f}, "
          f"val_loss: {val_loss:.4f}, val_acc: {val_acc:.4f}{marker}")
    print(f"  Val head losses: ring={val_heads['ring']:.4f}, asym={val_heads['asymmetry']:.4f}, "
          f"colour={val_heads['colour']:.4f}, elong={val_heads['elongation']:.4f}")

Epoch 1/15 - train_loss: 0.4873, train_acc: 0.7983, val_loss: 0.2445, val_acc: 0.9214 [NEW BEST]
  Val head losses: ring=0.0198, asym=0.0114, colour=0.0174, elong=0.0324
Epoch 2/15 - train_loss: 0.2127, train_acc: 0.9326, val_loss: 0.2105, val_acc: 0.9548 [NEW BEST]
  Val head losses: ring=0.0205, asym=0.0089, colour=0.0133, elong=0.0278
Epoch 3/15 - train_loss: 0.1465, train_acc: 0.9584, val_loss: 0.1485, val_acc: 0.9699 [NEW BEST]
  Val head losses: ring=0.0153, asym=0.0075, colour=0.0089, elong=0.0221
Epoch 4/15 - train_loss: 0.1085, train_acc: 0.9721, val_loss: 0.1373, val_acc: 0.9565 [NEW BEST]
  Val head losses: ring=0.0147, asym=0.0068, colour=0.0087, elong=0.0206
Epoch 5/15 - train_loss: 0.1112, train_acc: 0.9738, val_loss: 0.1381, val_acc: 0.9565
  Val head losses: ring=0.0128, asym=0.0058, colour=0.0090, elong=0.0228
Epoch 6/15 - train_loss: 0.1048, train_acc: 0.9742, val_loss: 0.1637, val_acc: 0.9783
  Val head losses: ring=0.0177, asym=0.0065, colour=0.0054, elong=0.0209
Ep

In [15]:
from sklearn.metrics import classification_report, confusion_matrix

eval_model = MultiHeadLensModel(timm.create_model('hf_hub:mwalmsley/zoobot-encoder-greyscale-convnext_nano', pretrained=False, in_chans=1, num_classes=0)).to(device)
eval_model.load_state_dict(torch.load('../models/multihead_convnext_nano_best.pt'))
eval_model.eval()

all_preds_mh = []
all_labels_mh = []

with torch.no_grad():
    for images_batch, labels_batch, heads_batch in test_loader:
        images_batch = images_batch.to(device)
        outputs = eval_model(images_batch)
        _, predicted = torch.max(outputs['class_logits'], 1)

        all_preds_mh.extend(predicted.cpu().numpy())
        all_labels_mh.extend(labels_batch.numpy())

print(classification_report(all_labels_mh, all_preds_mh, target_names=['non-lens', 'lens']))
print("Confusion matrix:")
print(confusion_matrix(all_labels_mh, all_preds_mh))

              precision    recall  f1-score   support

    non-lens       0.99      0.99      0.99       300
        lens       0.99      0.99      0.99       299

    accuracy                           0.99       599
   macro avg       0.99      0.99      0.99       599
weighted avg       0.99      0.99      0.99       599

Confusion matrix:
[[298   2]
 [  4 295]]
